# Exercise 2 — OpsTools

`build_ops_tools` returns a dict of four callable tools backed by a `TaskStore`.  The `executor_fn` is injectable — it replaces the real task execution logic so tests can run without any external system.

In [ ]:
import json
from dataclasses import dataclass, field

def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    text = str(text)
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except Exception:
        return None
@dataclass
class OpsTask:
    id:          str
    title:       str
    description: str = ""
    status:      str = "pending"
    result:      str = ""

class TaskStore:
    def __init__(self):
        self._tasks = {}; self._counter = 0
    def add(self, title, description=""):
        self._counter += 1
        tid = f"task_{self._counter:03d}"
        t = OpsTask(id=tid, title=title, description=description)
        self._tasks[tid] = t; return t
    def get(self, task_id): return self._tasks.get(task_id)
    def all(self): return list(self._tasks.values())
    def pending(self): return [t for t in self._tasks.values() if t.status == "pending"]
    def update(self, task_id, status, result=""):
        t = self._tasks.get(task_id)
        if t: t.status = status
        if t and result: t.result = result
        return t
    def __len__(self): return len(self._tasks)

# ── Exercise: implement build_ops_tools ──────────────────────────────────────

def build_ops_tools(store, executor_fn=None):
    """Build the four standard ops tools backed by a TaskStore.

    Returns {"list_tasks": fn, "run_task": fn,
             "check_status": fn, "generate_report": fn}
    """

    def list_tasks(status=None):
        # TODO: if status given, filter store.all(); otherwise return all.
        # Format each task as "[task_001] [pending] Title"
        # Return "No tasks found." if empty.
        return "No tasks found."

    def run_task(task_id):
        # TODO: get task; error if not found.
        # If status not in ("pending","failed"), return already-status message.
        # update to "running"; call executor_fn(task) or "Completed: <title>".
        # update to "done" with result; return "Task <id> done: <result>".
        # On exception: update to "failed" and return error.
        return "Not implemented"

    def check_status(task_id):
        # TODO: return "[task_id] title: status — result" (omit result if empty)
        return "Not implemented"

    def generate_report(scope="all"):
        # TODO: count tasks per status; return
        # "Ops Report (<scope>): <n> tasks — <k done, m pending, ...>"
        return "No tasks in store."

    return {
        "list_tasks":      list_tasks,
        "run_task":        run_task,
        "check_status":    check_status,
        "generate_report": generate_report,
    }


### Checks

In [ ]:
checks = 0
_exec = lambda t: "Simulated: " + t.title

# 1 — build_ops_tools returns dict with 4 keys
try:
    store = TaskStore()
    tools = build_ops_tools(store, executor_fn=_exec)
    assert set(tools.keys()) == {"list_tasks", "run_task", "check_status", "generate_report"}
    checks += 1; print("✅ 1 build_ops_tools returns dict with correct keys")
except Exception as e:
    print("❌ 1:", e)

# 2 — list_tasks shows all tasks
try:
    store = TaskStore()
    store.add("Task A"); store.add("Task B")
    tools = build_ops_tools(store)
    listing = tools["list_tasks"]()
    assert "Task A" in listing and "Task B" in listing
    checks += 1; print("✅ 2 list_tasks shows all tasks")
except Exception as e:
    print("❌ 2:", e)

# 3 — run_task marks task done and returns result
try:
    store = TaskStore()
    t = store.add("Lint check")
    tools = build_ops_tools(store, executor_fn=_exec)
    msg = tools["run_task"](t.id)
    assert "done" in msg.lower() or t.id in msg
    assert t.status == "done" and "Lint check" in t.result
    checks += 1; print("✅ 3 run_task marks task done with result")
except Exception as e:
    print("❌ 3:", e)

# 4 — check_status returns task status string
try:
    store = TaskStore()
    t = store.add("Deploy"); store.update(t.id, "done", "v2.0 deployed")
    tools = build_ops_tools(store)
    status = tools["check_status"](t.id)
    assert "done" in status.lower() and t.id in status
    checks += 1; print("✅ 4 check_status returns task status")
except Exception as e:
    print("❌ 4:", e)

# 5 — generate_report summarises the store
try:
    store = TaskStore()
    t1 = store.add("A"); t2 = store.add("B"); t3 = store.add("C")
    store.update(t1.id, "done", "ok"); store.update(t2.id, "done", "ok")
    tools = build_ops_tools(store)
    report = tools["generate_report"]()
    assert "3" in report or "three" in report.lower() or "2 done" in report or "done" in report
    assert "pending" in report.lower() or "1 pending" in report
    checks += 1; print("✅ 5 generate_report includes task counts by status")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
